In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
gas = pd.read_csv("Natural Gas Data.csv", skiprows=8, nrows=6, header=None)

In [ ]:
gas

In [ ]:
dates = gas.iloc[2, 1:]

In [ ]:
dates.head()

In [ ]:
residential = gas.iloc[4, 1:].str.replace(",", "").astype(float)

In [ ]:
residential.head()

In [ ]:
commercial = gas.iloc[5, 1:].str.replace(",", "").astype(float)

In [ ]:
commercial.head()

In [ ]:
cleanedgas = pd.DataFrame({"Date":dates,"Residential":residential,"Commercial":commercial})

In [ ]:
cleanedgas.head()

In [ ]:
cleanedgas["Total Demand"] = cleanedgas["Residential"]+cleanedgas["Commercial"]

In [ ]:
cleanedgas.head()

In [ ]:
weather = pd.read_csv("Temperature Data.csv", skiprows=3)

In [ ]:
weather.head()

In [ ]:
weather["HDD"] = np.maximum(0, 18 - weather["temperature_2m_mean (°C)"])

In [ ]:
weather.head()

In [ ]:
weather["time"] = pd.to_datetime(weather["time"])

In [ ]:
weather["Month"] = weather["time"].dt.to_period("M")

In [ ]:
monthly_hdd = weather.groupby("Month")["HDD"].sum().reset_index()

In [ ]:
monthly_hdd.head()

In [ ]:
cleanedgas["Month"] = pd.to_datetime(cleanedgas["Date"]).dt.to_period("M")

In [ ]:
cleanedgas.head()

In [ ]:
final_data = pd.merge(cleanedgas, monthly_hdd, on="Month")

In [ ]:
final_data.head()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import statsmodels.api as sm

In [ ]:
final_data["Year"] = final_data["Month"].dt.year
final_data["Month_Num"] = final_data["Month"].dt.month

In [ ]:
final_data["Winter"] = np.where(final_data["Month_Num"] >= 11,final_data["Year"] + 1,final_data["Year"])

In [ ]:
winter_data = final_data[final_data["Month_Num"].isin([11, 12, 1, 2, 3]) & final_data["Winter"].between(2017, 2025)]

In [ ]:
winter_data[["Month", "HDD", "Winter"]].head(10)

In [ ]:
winter_summary = winter_data.groupby("Winter")["HDD"].sum().reset_index()

In [ ]:
winter_summary

In [ ]:
coldest = winter_summary.loc[winter_summary["HDD"].idxmax()]

In [ ]:
mildest = winter_summary.loc[winter_summary["HDD"].idxmin()]

In [ ]:
coldest

In [ ]:
mildest

In [ ]:
average_hdd = winter_summary["HDD"].mean()

In [ ]:
average_hdd

In [ ]:
X_winter = winter_data[["HDD"]]
X_winter = sm.add_constant(X_winter)
y_winter = winter_data["Total Demand"]
winter_model = sm.OLS(y_winter, X_winter).fit()
print(winter_model.summary())

In [ ]:
scenarios = pd.DataFrame({"Scenario": ["Mild", "Average", "Cold"],"HDD": [mildest["HDD"], average_hdd, coldest["HDD"]]})

In [ ]:
scenarios["Predicted Demand (billion m³)"] = (winter_model.params["const"] * 5+ winter_model.params["HDD"] * scenarios["HDD"])/1_000_000

In [ ]:
scenarios

In [ ]:
mild_demand = scenarios.loc[scenarios["Scenario"] == "Mild", "Predicted Demand (billion m³)"].iloc[0]
cold_demand = scenarios.loc[scenarios["Scenario"] == "Cold", "Predicted Demand (billion m³)"].iloc[0]
demand_difference = cold_demand - mild_demand
percent_increase = (demand_difference/mild_demand)*100

print(f"Cold vs. Mild Difference: {demand_difference:.3f} billion m³")
print(f"Cold vs. Mild Increase: {percent_increase:.1f}%")

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(winter_data["HDD"],winter_data["Total Demand"]/1000000)
x_line = np.linspace(winter_data["HDD"].min(),winter_data["HDD"].max(),100)
y_line = (winter_model.params["const"]+ winter_model.params["HDD"] * x_line)/1000000
plt.plot(x_line, y_line, linestyle="--")
plt.title("Winter Gas Demand vs. Heating Degree Days (HDD)")
plt.xlabel("Heating Degree Days (HDD)")
plt.ylabel("Gas Demand (billion m³)")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
with pd.ExcelWriter("Natural_Gas_Demand_Analysis.xlsx") as writer:
    final_data.to_excel(writer, sheet_name="Monthly Data", index=False)
    scenarios.to_excel(writer, sheet_name="Winter Scenarios", index=False)